# Week 11 — Software engineering for economists

**Notebook outcomes**

- Organize a research project as a `uv`-managed Python package
- Use `ruff` to format and lint your code
- Write and run tests with `pytest`
- Write docstrings that double as runnable examples
- Use dependency groups, lockfiles, and workspaces in `uv`

We're not trying to make you a software engineer. We *are* trying to
save you from the research code that becomes unusable six months later
because nobody remembers how to run it.


## Why invest in this?

Three research-specific payoffs:

1. **Reproducibility** — your paper's results run years from now.
2. **Collaboration** — co-authors can clone and `uv sync`.
3. **Your future self** — opening last year's project doesn't mean
   spending two days unbreaking it.

The practices below are the cheapest possible investment for those
outcomes. Every one of them can be set up in under 10 minutes.


## Part 1 — Advanced `uv`

### Lockfiles recap

You already know `uv.lock` records exact versions. Two more tricks:

- **`uv sync`** — install the *exact* environment from the lockfile.
  On a new machine, this is all you need.
- **`uv lock --upgrade`** — update the lockfile to newer versions
  within your `pyproject.toml` constraints. Commit the result.

```sh
uv sync             # daily
uv lock --upgrade   # occasionally, intentionally
```


### Dependency groups

Not everything you need to work on a project needs to ship to every
user. Split into groups:

```toml
# pyproject.toml
[project]
dependencies = [
    "numpy>=2.0",
]

[dependency-groups]
dev = [
    "pytest>=8",
    "ruff>=0.8",
]
```

Install:

```sh
uv sync                         # prod deps only
uv sync --group dev             # + dev tools
```

In CI, use `--group dev`. In a Docker image used by collaborators in
production, skip it.


### Workspaces (preview)

If you have multiple related packages that depend on each other — say
a core library and a few paper-specific projects that use it — put them
in one `uv` **workspace**. One `uv.lock`, one set of tools, coherent
versions.

```toml
# root pyproject.toml
[tool.uv.workspace]
members = ["core", "paper_ajr", "paper_aiyagari"]
```

You won't need this in the camp. Know it's there for when a project
grows.


## Part 2 — `ruff`

`ruff` is a fast Python linter and formatter. You probably don't want
to argue about whitespace with your collaborators; `ruff` decides for
you.

### Install

It's already in your `dev` group. Run:

```sh
uv run ruff format .        # format every file
uv run ruff check .         # lint every file
uv run ruff check --fix .   # auto-fix simple issues
```

### Configure once

```toml
# pyproject.toml
[tool.ruff]
line-length = 100
extend-include = ["*.ipynb"]

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM"]
ignore = ["E501"]     # line length — we let formatter decide
```

Set your editor to run `ruff format` on save. Done.


## Part 3 — `pytest`

You've seen `assert` for ad-hoc tests. `pytest` is the industry-standard
tool that turns those into a full test suite.

### Setup

```
my_project/
├── pyproject.toml
├── src/
│   └── econtools/
│       ├── __init__.py
│       └── growth.py
└── tests/
    └── test_growth.py
```

### Writing a test

```python
# src/econtools/growth.py
def yoy_growth(prev: float, curr: float) -> float:
    """Year-over-year growth rate."""
    return (curr - prev) / prev


# tests/test_growth.py
from econtools.growth import yoy_growth


def test_positive_growth():
    assert yoy_growth(100, 110) == 0.10


def test_negative_growth():
    assert yoy_growth(200, 100) == -0.50


def test_zero_start_raises():
    import pytest
    with pytest.raises(ZeroDivisionError):
        yoy_growth(0, 100)
```

### Run the suite

```sh
uv run pytest
uv run pytest -v                # verbose
uv run pytest tests/test_growth.py::test_positive_growth   # one test
```

Tests passing turns into "green". One red test is all it takes to
notice a regression you didn't plan to make.


### Parametrizing tests

Run the same test body over multiple inputs:

```python
import pytest

@pytest.mark.parametrize("prev,curr,expected", [
    (100, 110, 0.10),
    (200, 100, -0.50),
    (50,  75,  0.50),
])
def test_yoy_growth(prev, curr, expected):
    assert yoy_growth(prev, curr) == pytest.approx(expected)
```

One test function, three test cases, three independent pass/fail
results. Very readable.


### What to test

For research code, prioritize:

1. **Core numerical routines** — if it's wrong, the paper is wrong.
2. **Edge cases** — zeros, empties, off-by-one.
3. **Known-answer cases** — a toy example you've solved by hand.

You don't need 100% coverage. You need coverage where it matters.


## Part 4 — Docstrings

A docstring is the string that follows a `def`, `class`, or at the top
of a module. Good ones include:

- One-line summary (imperative, ≤ 72 chars).
- A blank line.
- Longer description, if needed.
- Sections for parameters, returns, and examples.

```python
def yoy_growth(prev: float, curr: float) -> float:
    """Year-over-year growth rate.

    Parameters
    ----------
    prev : float
        Value in the previous period.
    curr : float
        Value in the current period.

    Returns
    -------
    float
        ``(curr - prev) / prev``.

    Examples
    --------
    >>> yoy_growth(100, 110)
    0.09999999999999998
    """
    return (curr - prev) / prev
```

The `>>> ...` block is a **doctest** — `pytest --doctest-modules` runs
it as a test. Your examples *and* your tests, in one place.


## Part 5 — Pre-commit hooks (optional)

Run lint and tests automatically when you `git commit`:

```
# .pre-commit-config.yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.8.0
    hooks:
      - id: ruff-format
      - id: ruff-check
```

Install: `uv run pre-commit install`. From then on, `git commit` runs
the hooks. Bad style, bad imports → commit refused until fixed.

This is a "set up once, get value forever" kind of tool.


## Project layout: a reference

A clean small research project looks like:

```
my_paper/
├── .gitignore
├── README.md
├── pyproject.toml
├── uv.lock
├── .pre-commit-config.yaml
├── src/
│   └── my_paper/
│       ├── __init__.py
│       ├── model.py
│       └── plots.py
├── tests/
│   └── test_model.py
├── notebooks/            # exploratory notebooks
│   └── 01_first_look.ipynb
├── paper/                # LaTeX source
│   ├── main.tex
│   └── refs.bib
└── data/                 # small data; big data lives elsewhere
    └── raw/
```

Boring and consistent is the goal. Anyone opening this repo knows where
to start (`README.md`), how to install (`pyproject.toml`), how to test
(`tests/`), and where the actual work lives (`src/`).


## Recap

- `uv sync` / `uv lock` + dependency groups + (later) workspaces.
- `ruff format` + `ruff check` — stop arguing about style.
- `pytest` for tests; `parametrize` for sweeps; docstrings can double as
  tests.
- Pre-commit hooks keep the above automatic.
- Standardize your project layout.
